# 14. SSE 流式恢复：怎样避免断线重连后的重复字、漏字与错终态？

## 面试回答主线

LLM 流式接口不能只返回一串匿名 `data`，每个事件都应带 request_id、单调 event_id、类型和可重放 payload。客户端收到事件后先按 ID 去重，再更新文本与终态游标；断线重连时通过 `Last-Event-ID` 请求服务端只重放后续事件。最简单的错误基线是重连后从头生成，它会把断线前的 token 再拼一次。服务端还必须面对日志保留窗口过短：如果游标已经过期，就不能假装无事发生，而应返回快照或明确要求重启。面试中我会用同一批真实中文增量复现重复文本，再手写 SSE 编解码、重放日志和幂等账本。最后说明代理缓冲、心跳、背压、鉴权与多副本日志才是生产系统真正的难点。

## 1. 真实案例：六条回答都在不同位置断线

每条会话包含一个 request_id、至少 4 个有序事件和断线发生在第几个事件之后。事件不是抽象数字，而是天气、订单、排障等真实回答片段；最后一个事件使用 `done` 类型承载终态。我们先展示断线前已落账的文本和客户端应携带的最后事件 ID。

In [1]:
from pprint import pprint  # 导入结构化打印工具以查看流式事件样本
sessions = [{"request_id": "req-weather", "chunks": ["北京", "今天", "晴，", "最高 29℃"], "cut": 2}, {"request_id": "req-order", "chunks": ["订单 A17", "已经出库，", "预计明天", "送达"], "cut": 1}, {"request_id": "req-code", "chunks": ["空指针来自", "未初始化的 cache，", "请在读取前", "建立默认值"], "cut": 3}, {"request_id": "req-rag", "chunks": ["检索命中", "退款规则第 4 条，", "答案需要", "附带来源"], "cut": 2}, {"request_id": "req-agent", "chunks": ["已查询库存，", "上海仓有 12 件，", "可以创建", "调拨单"], "cut": 1}, {"request_id": "req-summary", "chunks": ["会议决定", "周五灰度发布，", "负责人是小林，", "回滚阈值为 2%"], "cut": 3}]  # 定义六个带中文语义和断点的真实流式会话
def make_events(session):  # 把一个会话的文本增量转换成可重放事件
    events = []  # 初始化当前请求的追加日志
    for index, chunk in enumerate(session["chunks"], start=1):  # 为每个文本增量分配单调序号
        event_type = "done" if index == len(session["chunks"]) else "delta"  # 将最后一个增量标记为终态事件
        events.append({"id": f'{session["request_id"]}:{index}', "request_id": session["request_id"], "seq": index, "event": event_type, "data": chunk})  # 保存完整事件信封
    return events  # 返回当前会话的有序事件日志
event_logs = {session["request_id"]: make_events(session) for session in sessions}  # 为六条会话建立服务端重放日志
preview = [{"请求": session["request_id"], "断线后游标": event_logs[session["request_id"]][session["cut"] - 1]["id"], "已收文本": "".join(session["chunks"][:session["cut"]])} for session in sessions]  # 汇总断线瞬间的客户端状态
print("SSE 断线样本预览：")  # 输出输入预览标题
pprint(preview, sort_dicts=False)  # 展示六个真实请求的断点和已收内容

SSE 断线样本预览：
[{'请求': 'req-weather', '断线后游标': 'req-weather:2', '已收文本': '北京今天'},
 {'请求': 'req-order', '断线后游标': 'req-order:1', '已收文本': '订单 A17'},
 {'请求': 'req-code', '断线后游标': 'req-code:3', '已收文本': '空指针来自未初始化的 cache，请在读取前'},
 {'请求': 'req-rag', '断线后游标': 'req-rag:2', '已收文本': '检索命中退款规则第 4 条，'},
 {'请求': 'req-agent', '断线后游标': 'req-agent:1', '已收文本': '已查询库存，'},
 {'请求': 'req-summary', '断线后游标': 'req-summary:3', '已收文本': '会议决定周五灰度发布，负责人是小林，'}]


## 2. Baseline（基线）：重连后从第一个 token 重新拼接

这个基线很常见：服务器重新执行请求并从头发送，客户端又把新流直接追加到旧文本。它没有事件 ID，也没有幂等账本，因此每条会话都会重复断线前的前缀。下面在同一批会话上真实拼出错误文本，并与期望文本逐条比较。

In [2]:
baseline_rows = []  # 收集从头重放基线的逐请求错误
for session in sessions:  # 遍历所有断线会话
    before_disconnect = "".join(session["chunks"][:session["cut"]])  # 模拟客户端断线前已经显示的文本
    restarted_stream = "".join(session["chunks"])  # 模拟服务端重连后从头再次生成的完整文本
    baseline_text = before_disconnect + restarted_stream  # 复现客户端无去重直接追加造成的重复前缀
    expected_text = "".join(session["chunks"])  # 构造没有重复和漏字的业务期望
    baseline_rows.append({"请求": session["request_id"], "错误结果": baseline_text, "期望": expected_text, "重复字符数": len(baseline_text) - len(expected_text)})  # 保存每个请求的可观察错误
print("无游标重连的基线结果：")  # 标注当前输出属于错误基线
pprint(baseline_rows, sort_dicts=False)  # 展示每条中文回答如何产生重复文本

无游标重连的基线结果：
[{'请求': 'req-weather',
  '错误结果': '北京今天北京今天晴，最高 29℃',
  '期望': '北京今天晴，最高 29℃',
  '重复字符数': 4},
 {'请求': 'req-order',
  '错误结果': '订单 A17订单 A17已经出库，预计明天送达',
  '期望': '订单 A17已经出库，预计明天送达',
  '重复字符数': 6},
 {'请求': 'req-code',
  '错误结果': '空指针来自未初始化的 cache，请在读取前空指针来自未初始化的 cache，请在读取前建立默认值',
  '期望': '空指针来自未初始化的 cache，请在读取前建立默认值',
  '重复字符数': 22},
 {'请求': 'req-rag',
  '错误结果': '检索命中退款规则第 4 条，检索命中退款规则第 4 条，答案需要附带来源',
  '期望': '检索命中退款规则第 4 条，答案需要附带来源',
  '重复字符数': 14},
 {'请求': 'req-agent',
  '错误结果': '已查询库存，已查询库存，上海仓有 12 件，可以创建调拨单',
  '期望': '已查询库存，上海仓有 12 件，可以创建调拨单',
  '重复字符数': 6},
 {'请求': 'req-summary',
  '错误结果': '会议决定周五灰度发布，负责人是小林，会议决定周五灰度发布，负责人是小林，回滚阈值为 2%',
  '期望': '会议决定周五灰度发布，负责人是小林，回滚阈值为 2%',
  '重复字符数': 18}]


## 3. 手写核心算法：SSE 帧、Last-Event-ID 与追加日志

SSE 帧至少包含 `id`、`event` 和 JSON `data`，空行表示一帧结束。这里手写编码与解析，确保学习者能看到线上字节协议，而不是调用高耦合客户端。服务端恢复函数在日志中找到 last_id，只返回它之后的事件；首次连接则返回全部。

In [3]:
import json  # 导入标准 JSON 编解码器以构造 SSE 数据行
def encode_sse(event):  # 将结构化事件编码为标准 SSE 文本帧
    payload = json.dumps({"request_id": event["request_id"], "seq": event["seq"], "text": event["data"]}, ensure_ascii=False)  # 把业务载荷序列化为可传输 JSON
    return f'id: {event["id"]}\nevent: {event["event"]}\ndata: {payload}\n\n'  # 按 SSE 行协议生成带空行终止符的帧
def parse_sse(frame):  # 手写解析一个完整 SSE 文本帧
    fields = {}  # 初始化帧字段字典
    for line in frame.strip().splitlines():  # 逐行解析 id、event 和 data
        name, value = line.split(":", 1)  # 只在第一个冒号处分离字段名和值
        fields[name] = value.lstrip()  # 去除协议允许的单个前导空格并保存字段
    payload = json.loads(fields["data"])  # 把 data 行恢复为结构化业务载荷
    return {"id": fields["id"], "event": fields["event"], **payload}  # 合并协议字段与业务字段供客户端消费
def replay_after(events, last_event_id):  # 根据客户端游标选择需要重放的日志后缀
    if last_event_id is None:  # 首次连接没有历史游标
        return list(events)  # 首次连接需要发送完整事件日志
    positions = {event["id"]: index for index, event in enumerate(events)}  # 建立事件 ID 到日志位置的索引
    if last_event_id not in positions:  # 检测客户端游标是否仍在保留窗口内
        raise ValueError("resume_cursor_expired")  # 明确报告过期而不是静默漏发
    return events[positions[last_event_id] + 1:]  # 只重放最后确认事件之后的增量
sample_event = event_logs["req-weather"][0]  # 选择第一条天气增量演示线上帧
sample_frame = encode_sse(sample_event)  # 编码一帧真实中文 SSE 消息
parsed_event = parse_sse(sample_frame)  # 再解析该帧验证协议字段能够往返
print("一帧真实 SSE 文本：")  # 输出协议示例标题
print(sample_frame)  # 展示浏览器实际收到的多行帧
print("解析后的字段：", parsed_event)  # 展示事件 ID、类型和中文载荷

一帧真实 SSE 文本：
id: req-weather:1
event: delta
data: {"request_id": "req-weather", "seq": 1, "text": "北京"}


解析后的字段： {'id': 'req-weather:1', 'event': 'delta', 'request_id': 'req-weather', 'seq': 1, 'text': '北京'}


## 4. 幂等客户端账本：先去重，再更新可见文本

客户端账本以 event_id 为唯一键。即使网络层重复投递同一帧，消费函数也不会再次拼接；只有 `done` 事件才能把请求标为完成。下面先消费断线前事件，再用 Last-Event-ID 拉取后缀，最后故意重复投递一次以验证幂等。

In [4]:
def consume(ledger, event):  # 定义客户端对单个事件的幂等消费逻辑
    if event["id"] in ledger["seen"]:  # 检查该事件是否已经提交到账本
        return False  # 重复投递不再改变文本和终态
    ledger["seen"].add(event["id"])  # 先记录事件 ID 以形成幂等边界
    ledger["text"] += event["data"]  # 只把首次出现的文本增量追加到界面
    ledger["last_id"] = event["id"]  # 推进可以用于下次恢复的确认游标
    ledger["done"] = event["event"] == "done"  # 仅根据显式终态事件更新完成状态
    return True  # 告知调用方本次事件确实产生了状态变化
resume_ledgers = {}  # 保存六条会话恢复后的客户端账本
for session in sessions:  # 逐条模拟断线与恢复全过程
    ledger = {"seen": set(), "text": "", "last_id": None, "done": False}  # 初始化当前请求的本地幂等状态
    events = event_logs[session["request_id"]]  # 读取当前请求的服务端追加日志
    for event in events[:session["cut"]]:  # 消费断线前已经到达的事件前缀
        consume(ledger, event)  # 将首次事件提交到客户端账本
    resumed_events = replay_after(events, ledger["last_id"])  # 携带最后事件 ID 获取尚未收到的后缀
    for event in resumed_events:  # 依序消费服务端重放的剩余事件
        consume(ledger, event)  # 更新文本、游标和完成标志
    consume(ledger, resumed_events[-1])  # 故意重复投递最后一帧以验证不会重复拼字
    resume_ledgers[session["request_id"]] = ledger  # 保存当前请求的最终账本用于结果比较
print("天气请求恢复后的幂等账本：")  # 标注中间状态示例
print({key: value for key, value in resume_ledgers["req-weather"].items() if key != "seen"}, "已见事件数=", len(resume_ledgers["req-weather"]["seen"]))  # 展示文本、游标、终态与去重规模

天气请求恢复后的幂等账本：
{'text': '北京今天晴，最高 29℃', 'last_id': 'req-weather:4', 'done': True} 已见事件数= 4


## 5. 结果解读：逐请求检查重复、漏字和终态

正确方案的目标不只是“能继续输出”，而是文本等于期望、事件数等于唯一日志长度、最终状态为 done。对照表保留基线重复字符数，便于量化恢复协议解决了什么问题。

In [5]:
result_rows = []  # 构造六条会话的最终对照表
for session, baseline in zip(sessions, baseline_rows):  # 对齐原始会话、错误基线和修正账本
    ledger = resume_ledgers[session["request_id"]]  # 读取当前请求恢复后的客户端状态
    expected = "".join(session["chunks"])  # 重新构造该请求的完整期望回答
    result_rows.append({"请求": session["request_id"], "基线重复字符": baseline["重复字符数"], "恢复文本": ledger["text"], "无重复漏字": ledger["text"] == expected, "收到终态": ledger["done"]})  # 保存逐请求可审计结论
print("SSE 恢复方案逐样本结果：")  # 输出结果解读标题
pprint(result_rows, sort_dicts=False)  # 展示每个请求的文本完整性和终态

SSE 恢复方案逐样本结果：
[{'请求': 'req-weather',
  '基线重复字符': 4,
  '恢复文本': '北京今天晴，最高 29℃',
  '无重复漏字': True,
  '收到终态': True},
 {'请求': 'req-order',
  '基线重复字符': 6,
  '恢复文本': '订单 A17已经出库，预计明天送达',
  '无重复漏字': True,
  '收到终态': True},
 {'请求': 'req-code',
  '基线重复字符': 22,
  '恢复文本': '空指针来自未初始化的 cache，请在读取前建立默认值',
  '无重复漏字': True,
  '收到终态': True},
 {'请求': 'req-rag',
  '基线重复字符': 14,
  '恢复文本': '检索命中退款规则第 4 条，答案需要附带来源',
  '无重复漏字': True,
  '收到终态': True},
 {'请求': 'req-agent',
  '基线重复字符': 6,
  '恢复文本': '已查询库存，上海仓有 12 件，可以创建调拨单',
  '无重复漏字': True,
  '收到终态': True},
 {'请求': 'req-summary',
  '基线重复字符': 18,
  '恢复文本': '会议决定周五灰度发布，负责人是小林，回滚阈值为 2%',
  '无重复漏字': True,
  '收到终态': True}]


## 6. 失败案例与修正：Last-Event-ID 已超出日志保留窗口

如果服务端只保留最近两帧，而客户端拿着第一帧游标回来，直接发送保留窗口会漏掉中间文本。正确做法是检测游标不存在，返回带新游标的权威快照，或者用 409/410 明确要求整次重启；绝不能把缺口伪装成正常恢复。这里实现“快照覆盖本地派生文本”的修正路径。

In [6]:
stale_session = sessions[0]  # 选择天气会话复现游标过期故障
full_log = event_logs[stale_session["request_id"]]  # 读取该请求的完整追加日志
retained_log = full_log[-2:]  # 模拟服务端只保留最近两个事件
stale_cursor = full_log[0]["id"]  # 模拟客户端携带已经被淘汰的旧游标
failure_message = ""  # 初始化游标过期的显式错误信息
try:  # 尝试从不完整保留窗口继续重放
    replay_after(retained_log, stale_cursor)  # 使用旧游标查询只剩两帧的日志
except ValueError as error:  # 捕获服务端检测到的恢复缺口
    failure_message = str(error)  # 保存可观测错误码供协议层返回
snapshot = {"request_id": stale_session["request_id"], "text": "".join(stale_session["chunks"]), "last_id": full_log[-1]["id"], "done": True}  # 构造可覆盖本地状态的权威快照
print({"失败复现": failure_message, "错误做法会漏掉": full_log[1]["data"], "修正快照": snapshot})  # 同时展示缺口和快照修复内容

{'失败复现': 'resume_cursor_expired', '错误做法会漏掉': '今天', '修正快照': {'request_id': 'req-weather', 'text': '北京今天晴，最高 29℃', 'last_id': 'req-weather:4', 'done': True}}


## 7. 生产差距与最小回归检查

真实服务还要处理 Nginx/CDN 缓冲、心跳保活、浏览器重连退避、慢消费者背压、请求鉴权和多副本共享日志。事件 ID 必须在请求内单调且不可复用，日志保留时间应覆盖客户端最大重连窗口；快照也要有版本号以避免旧快照覆盖新状态。下面的少量断言只确认本实验已经展示的文本完整性、幂等性、终态和过期游标分支。

In [7]:
assert len(sessions) >= 5  # 确认断线案例数量足以进行逐样本观察
assert all(row["重复字符数"] > 0 for row in baseline_rows)  # 确认从头重放基线真实产生了重复文本
assert all(row["无重复漏字"] for row in result_rows)  # 确认 Last-Event-ID 恢复得到完整且唯一的文本
assert all(row["收到终态"] for row in result_rows)  # 确认每条恢复会话都消费到显式 done 事件
assert failure_message == "resume_cursor_expired"  # 确认过期游标被识别而非静默漏发
assert snapshot["text"] == "".join(stale_session["chunks"])  # 确认权威快照能完整修复客户端状态
print("回归检查通过：断线恢复、事件去重、终态提交与快照修复均符合预期。")  # 输出最终验收结论

回归检查通过：断线恢复、事件去重、终态提交与快照修复均符合预期。
